In [3]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

from langchain_groq import ChatGroq

In [23]:
load_dotenv()
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [6]:
loader=PyPDFLoader("Medical_book.pdf")
documents=loader.load()

In [8]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=splitter.split_documents(documents)

In [11]:
embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
pc=Pinecone(api_key=PINECONE_API_KEY)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22880.85it/s]


In [12]:
index_name = "medical-chatbot"

if index_name not in pc.list_indexes().names():
    pc.create_index( name=index_name, dimension=384, metric="cosine",
                     spec=ServerlessSpec(cloud="aws",region="us-east-1"))

In [13]:
vectorstore=PineconeVectorStore.from_documents(documents=chunks,embedding=embedding,
                                               index_name=index_name)

In [14]:
retriever=vectorstore.as_retriever(search_kwargs={"k":3})
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0)

In [15]:
prompt=ChatPromptTemplate.from_template("""
You are a medical assistant.

Answer ONLY from the provided context.

If the answer is unavailable, reply:

"I don't know."

Context:
{context}

Question:
{question}
""")

In [21]:
def ask_question():
    question=input("Ask your question: ")
    docs=retriever.invoke(question)
    context="\n\n".join(doc.page_content for doc in docs)

    messages = prompt.invoke({"context": context,"question": question})
    response = llm.invoke(messages)

    print("\nAnswer:\n")
    print(response.content)

In [24]:
ask_question()

Ask your question:  what is cancer ??



Answer:

Cancer is characterized by the development of malignant cells, which are cells that have uncontrolled division leading to abnormal growth and the ability to invade normal tissue locally or to spread throughout the body, in a process called metastasis.
